In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project Roaot:", PROJECT_ROOT)

Project Root: /Users/dhairyas87/Documents/Projects/explainable-credit-risk-management/xai-credit-risk-freddie-mac


In [3]:
import importlib

import scripts.build_target_dataset
import scripts.pipeline
import scripts.build_feature_store

importlib.reload(scripts.build_target_dataset)
importlib.reload(scripts.build_feature_store)
importlib.reload(scripts.pipeline)

<module 'scripts.pipeline' from '/Users/dhairyas87/Documents/Projects/explainable-credit-risk-management/xai-credit-risk-freddie-mac/scripts/pipeline.py'>

In [30]:
import scripts.build_target_dataset as btd

dir(btd)

['Path',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'build_target_dataset',
 'create_bssi',
 'create_loan_level_performance',
 'create_stress_flag',
 'load_origination',
 'load_performance',
 'validate_loan_counts']

In [10]:
from src.schema import ORIGINATION_COLUMNS

In [11]:
ORIGINATION_COLUMNS

['credit_score',
 'first_payment_date',
 'first_time_homebuyer_indicator',
 'maturity_date',
 'msa',
 'mortgage_insurance_pct',
 'num_units',
 'occupancy_status',
 'cltv',
 'dti',
 'original_upb',
 'ltv',
 'interest_rate',
 'channel',
 'prepayment_penalty_indicator',
 'amortization_type',
 'property_state',
 'property_type',
 'postal_code',
 'loan_identifier',
 'loan_purpose',
 'original_loan_term',
 'num_borrowers',
 'seller_name',
 'servicer_name',
 'super_conforming_flag',
 'pre_harp_loan_sequence_number',
 'program_indicator',
 'harp_indicator',
 'property_valuation_method',
 'interest_only_indicator',
 'mortgage_insurance_cancellation_indicator']

In [12]:
from scripts.pipeline import run_pipeline

QUARTERS = [
    "2018Q1",
    "2018Q2",
    "2018Q3",
    "2018Q4"
]

for quarter in QUARTERS:

    run_pipeline(
        quarter=quarter,
        force_rebuild=True
    )

RUNNING PIPELINE : 2018Q1

Generating target dataset...
Loading data...
Running validation...
Origination Loans : 296816
Performance Loans : 296816
Validation Passed
Creating loan-level dataset...
Saving parquet...
Done

Generating master dataset...
BUILDING MASTER DATASET

Loading origination data...
Origination shape: (296816, 32)

Loading target dataset...
Target shape: (296816, 7)

Cleaning origination data...

Merging datasets...
Master shape after merge: (296816, 38)

Creating Borrower Serviceability Score...

Validating dataset...
Master dataset validation passed.

Saving dataset...

Dataset saved successfully.
Final shape: (296816, 46)

Pipeline completed.
RUNNING PIPELINE : 2018Q2

Generating target dataset...
Loading data...
Running validation...
Origination Loans : 364502
Performance Loans : 364502
Validation Passed
Creating loan-level dataset...
Saving parquet...
Done

Generating master dataset...
BUILDING MASTER DATASET

Loading origination data...
Origination shape: (3645

In [31]:
from scripts.pipeline import run_pipeline

QUARTERS = [
    "2018Q1",
    "2018Q2",
    "2018Q3",
    "2018Q4"
]

for quarter in QUARTERS:

    run_pipeline(
        quarter=quarter,
        force_rebuild=True
    )

RUNNING PIPELINE : 2018Q1

Generating target dataset...
Loading data...
Running validation...
Origination Loans : 296816
Performance Loans : 296816
Validation Passed
Creating loan-level dataset...
Saving parquet...
Done

Generating master dataset...
BUILDING MASTER DATASET

Loading origination data...
Origination shape: (296816, 32)

Loading target dataset...
Target shape: (296816, 7)

Cleaning origination data...

Merging datasets...
Master shape after merge: (296816, 38)

Creating Borrower Serviceability Score...

Validating dataset...
Master dataset validation passed.

Saving dataset...

Dataset saved successfully.
Final shape: (296816, 46)

Pipeline completed.
RUNNING PIPELINE : 2018Q2

Generating target dataset...
Loading data...
Running validation...
Origination Loans : 364502
Performance Loans : 364502
Validation Passed
Creating loan-level dataset...
Saving parquet...
Done

Generating master dataset...
BUILDING MASTER DATASET

Loading origination data...
Origination shape: (3645

In [14]:
from scripts.build_combined_dataset import (
    build_combined_dataset
)

combined_df = build_combined_dataset(
    input_files=[
        "../data/processed/master_dataset_2018Q1.parquet",
        "../data/processed/master_dataset_2018Q2.parquet",
        "../data/processed/master_dataset_2018Q3.parquet",
        "../data/processed/master_dataset_2018Q4.parquet"
    ],
    output_path="../data/processed/master_dataset_2018.parquet"
)

Loading ../data/processed/master_dataset_2018Q1.parquet
Loading ../data/processed/master_dataset_2018Q2.parquet
Loading ../data/processed/master_dataset_2018Q3.parquet
Loading ../data/processed/master_dataset_2018Q4.parquet
Combined Shape: (1285434, 49)
Saved: ../data/processed/master_dataset_2018.parquet


In [15]:
from scripts.build_feature_store import (
    build_feature_store
)

build_feature_store(
    input_path="../data/processed/master_dataset_2018.parquet",
    output_dir="../data/modeling"
)

BUILDING FEATURE STORE

Loading dataset...
Dataset Shape: (1285434, 49)

Creating baseline dataset...
Saved baseline datasets.
Baseline Shape: (1285434, 35)

Creating BSS dataset...
Saved bss datasets.
BSS Shape: (1285434, 38)

Feature store creation complete.


In [16]:
import pandas as pd

pd.read_parquet(
    "../data/modeling/train_baseline.parquet"
).shape



(661318, 35)

In [17]:
import pandas as pd
pd.read_parquet(
    "../data/modeling/train_bss.parquet"
).shape

(661318, 38)

In [18]:
for name in [
    "train_baseline",
    "valid_baseline",
    "test_baseline"
]:

    df = pd.read_parquet(
        f"../data/modeling/{name}.parquet"
    )

    print(
        name,
        df.shape
    )

train_baseline (661318, 35)
valid_baseline (336669, 35)
test_baseline (287447, 35)


In [19]:
for name in [
    "train_baseline",
    "valid_baseline",
    "test_baseline"
]:

    df = pd.read_parquet(
        f"../data/modeling/{name}.parquet"
    )

    print(
        name,
        round(
            df["stress_flag"].mean() * 100,
            2
        )
    )

train_baseline 13.97
valid_baseline 13.74
test_baseline 13.39


In [20]:
train_df = pd.read_parquet(
    "../data/modeling/train_baseline.parquet"
)

print(train_df.shape)

train_df.columns.tolist()

(661318, 35)


['credit_score',
 'first_payment_date',
 'first_time_homebuyer_indicator',
 'maturity_date',
 'msa',
 'mortgage_insurance_pct',
 'num_units',
 'occupancy_status',
 'cltv',
 'dti',
 'original_upb',
 'ltv',
 'interest_rate',
 'channel',
 'prepayment_penalty_indicator',
 'amortization_type',
 'property_state',
 'property_type',
 'loan_identifier',
 'loan_purpose',
 'original_loan_term',
 'num_borrowers',
 'seller_name',
 'servicer_name',
 'super_conforming_flag',
 'program_indicator',
 'harp_indicator',
 'property_valuation_method',
 'interest_only_indicator',
 'mortgage_insurance_cancellation_indicator',
 'bssi',
 'stress_flag',
 'year',
 'quarter',
 'quarter_num']

In [21]:
master_df = pd.read_parquet(
    "../data/processed/master_dataset_2018.parquet"
)

print(master_df.columns.tolist())

['credit_score', 'first_payment_date', 'first_time_homebuyer_indicator', 'maturity_date', 'msa', 'mortgage_insurance_pct', 'num_units', 'occupancy_status', 'cltv', 'dti', 'original_upb', 'ltv', 'interest_rate', 'channel', 'prepayment_penalty_indicator', 'amortization_type', 'property_state', 'property_type', 'postal_code', 'loan_identifier', 'loan_purpose', 'original_loan_term', 'num_borrowers', 'seller_name', 'servicer_name', 'super_conforming_flag', 'pre_harp_loan_sequence_number', 'program_indicator', 'harp_indicator', 'property_valuation_method', 'interest_only_indicator', 'mortgage_insurance_cancellation_indicator', 'max_delinquency', 'ever_ra', 'ever_modified', 'ever_assistance', 'bssi', 'stress_flag', 'credit_score_norm', 'dti_norm', 'interest_rate_norm', 'ltv_norm', 'num_borrowers_norm', 'bss', 'bss_bucket', 'bss_level', 'year', 'quarter', 'quarter_num']


In [22]:
q1 = pd.read_parquet(
    "../data/processed/master_dataset_2018Q1.parquet"
)

print(q1.columns.tolist())

['credit_score', 'first_payment_date', 'first_time_homebuyer_indicator', 'maturity_date', 'msa', 'mortgage_insurance_pct', 'num_units', 'occupancy_status', 'cltv', 'dti', 'original_upb', 'ltv', 'interest_rate', 'channel', 'prepayment_penalty_indicator', 'amortization_type', 'property_state', 'property_type', 'postal_code', 'loan_identifier', 'loan_purpose', 'original_loan_term', 'num_borrowers', 'seller_name', 'servicer_name', 'super_conforming_flag', 'pre_harp_loan_sequence_number', 'program_indicator', 'harp_indicator', 'property_valuation_method', 'interest_only_indicator', 'mortgage_insurance_cancellation_indicator', 'max_delinquency', 'ever_ra', 'ever_modified', 'ever_assistance', 'bssi', 'stress_flag', 'credit_score_norm', 'dti_norm', 'interest_rate_norm', 'ltv_norm', 'num_borrowers_norm', 'bss', 'bss_bucket', 'bss_level']


In [23]:
for col in [
    "seller_name",
    "servicer_name",
    "msa"
]:
    print(
        col,
        train_df[col].nunique()
    )

seller_name 25
servicer_name 24
msa 447
